In [0]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from mlflow.models.signature import infer_signature

df_pd = spark.read.table("proyecto_gestion_costos_operativos.default.gold_equipos_features").toPandas()
features = ['Price_X', 'Price_Y', 'Price_Z', 'Price_X_lag30', 'Price_Y_lag30', 'Price_Z_lag30']


def entrenar_modelo_explicativo(target_col, nombre_run):
    """
    CORRECCIÓN: se cambia de RandomForestClassifier (prediciendo subida/bajada
    binaria) a RandomForestRegressor (prediciendo el precio real del equipo).

    Motivo: el objetivo de este notebook es identificar qué insumos EXPLICAN
    el comportamiento del precio y cuáles son ruido -- no es un modelo de
    pronóstico (esa tarea la hace Prophet en el Notebook 3, que sí es un
    modelo de series de tiempo). Para ese objetivo explicativo, clasificar
    "sube/no sube" descarta la magnitud del cambio y responde una pregunta
    más pobre que la que realmente interesa a la gerencia. Un modelo de
    regresión sobre el precio real da una medida de importancia de variable
    (feature_importances_) directamente ligada a cuánto pesa cada insumo en
    el NIVEL del precio, que es la pregunta de negocio real.
    """
    X = df_pd[features]
    y = df_pd[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    with mlflow.start_run(run_name=nombre_run):
        model = RandomForestRegressor(n_estimators=200, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        # Métricas de regresión (reemplazan a las métricas de clasificación)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # squared=False fue removido en scikit-learn 1.4+
        r2 = r2_score(y_test, y_pred)

        mlflow.log_metrics({
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

        X_test_con_target = X_test.copy()
        X_test_con_target[target_col] = y_test
        dataset_prueba = mlflow.data.from_pandas(X_test_con_target, targets=target_col)
        mlflow.log_input(dataset_prueba, context="testing")

        # Separación de Señal vs Ruido: importancia de variable sobre el precio real
        importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
        print(f"\n=== Variables determinantes para {target_col} (R2 test = {r2:.3f}) ===")
        print(importances)

        signature = infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=f"modelo_regresion_{target_col}",
            signature=signature,
            input_example=X_train.head(5)
        )

        return importances, {"MAE": mae, "RMSE": rmse, "R2": r2}


# Se entrena y se compara señal vs ruido para AMBOS equipos por separado
importancias_eq1, metricas_eq1 = entrenar_modelo_explicativo("Price_Equipo1", "Explicativo_Precio_Equipo1")
importancias_eq2, metricas_eq2 = entrenar_modelo_explicativo("Price_Equipo2", "Explicativo_Precio_Equipo2")

comparacion_importancias = pd.DataFrame({
    "Equipo_1": importancias_eq1,
    "Equipo_2": importancias_eq2
}).sort_values("Equipo_1", ascending=False)

print("\n=== Comparación de variables determinantes por equipo ===")
print(comparacion_importancias)

print("\n=== Calidad del modelo explicativo (R2) ===")
print(f"Equipo 1: {metricas_eq1}")
print(f"Equipo 2: {metricas_eq2}")

# Se guarda como tabla Delta para que el agente de IA (Notebook 4) pueda
# citarla si le preguntan "por qué" un insumo importa más que otro
spark.createDataFrame(
    comparacion_importancias.reset_index().rename(columns={"index": "Variable"})
).write.format("delta").mode("overwrite").saveAsTable(
    "proyecto_gestion_costos_operativos.default.importancia_variables_por_equipo"
)

2026/07/29 18:39:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Variables determinantes para Price_Equipo1 (R2 test = 0.682) ===
Price_Y          0.991642
Price_X          0.002855
Price_X_lag30    0.001793
Price_Z          0.001545
Price_Z_lag30    0.001487
Price_Y_lag30    0.000678
dtype: float64


🔗 View Logged Model at: https://adb-7405618155100520.0.azuredatabricks.net/ml/experiments/2450753263925787/models/m-27cea2c0de1049f79948df97ec6e5c3a?o=7405618155100520
2026/07/29 18:39:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



=== Variables determinantes para Price_Equipo2 (R2 test = 0.595) ===
Price_Z          0.957527
Price_Y          0.024649
Price_X          0.009763
Price_X_lag30    0.003157
Price_Z_lag30    0.002520
Price_Y_lag30    0.002385
dtype: float64


🔗 View Logged Model at: https://adb-7405618155100520.0.azuredatabricks.net/ml/experiments/2450753263925787/models/m-0a5ead3d762f462ca7eb32797d1eb518?o=7405618155100520



=== Comparación de variables determinantes por equipo ===
               Equipo_1  Equipo_2
Price_Y        0.991642  0.024649
Price_X        0.002855  0.009763
Price_X_lag30  0.001793  0.003157
Price_Z        0.001545  0.957527
Price_Z_lag30  0.001487  0.002520
Price_Y_lag30  0.000678  0.002385

=== Calidad del modelo explicativo (R2) ===
Equipo 1: {'MAE': 38.948157428571335, 'RMSE': np.float64(62.42342564060059), 'R2': 0.6824670090496103}
Equipo 2: {'MAE': 62.37505207142874, 'RMSE': np.float64(101.18741814732417), 'R2': 0.5949275888345011}
